### Etapa 1 — Validar ambiente e arquivos

In [ ]:
from pathlib import Path
import sys
import pandas as pd

# Raiz do repositório
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"

print("Python:", sys.executable)
print("Raiz do projeto:", ROOT)
print("\nArquivos de entrada:")

for nome in ["fortigate_syslog.log", "ICMP_Data_exfiltration.log.gz"]:
    arquivo = DATA / nome
    print(f"{nome}: {'OK' if arquivo.exists() else 'NÃO ENCONTRADO'}")
    if arquivo.exists():
        print(f"  Tamanho: {arquivo.stat().st_size:,} bytes")

print("\nPandas:", pd.__version__)

### Etapa 2 — Inspecionar o formato dos logs

In [ ]:
import gzip

arquivos = {
    "FortiGate": DATA / "fortigate_syslog.log",
    "ICMP comprimido": DATA / "ICMP_Data_exfiltration.log.gz",
}

for nome, caminho in arquivos.items():
    print(f"\n=== {nome} ===")

    abrir = gzip.open if caminho.suffix == ".gz" else open

    with abrir(caminho, "rt", encoding="utf-8", errors="replace") as f:
        for i, linha in enumerate(f, start=1):
            print(linha.rstrip()[:300])
            if i == 3:
                break

In [ ]:
import hashlib
import gzip

def hash_conteudo(caminho):
    abrir = gzip.open if caminho.suffix == ".gz" else open
    sha256 = hashlib.sha256()
    linhas = 0

    with abrir(caminho, "rb") as f:
        for linha in f:
            sha256.update(linha)
            linhas += 1

    return linhas, sha256.hexdigest()

for nome, caminho in arquivos.items():
    linhas, hash_sha = hash_conteudo(caminho)
    print(f"{nome}:")
    print(f"  Linhas: {linhas:,}")
    print(f"  SHA-256: {hash_sha}")

### Etapa 3 — Mapear os eventos do FortiGate

In [ ]:
import re
import pandas as pd

registros = []

with (DATA / "fortigate_syslog.log").open(
    encoding="utf-8", errors="replace"
) as arquivo:
    for linha in arquivo:
        campos = dict(
            re.findall(r'(\w+)=("[^"]*"|\S+)', linha)
        )
        campos = {
            chave: valor.strip('"')
            for chave, valor in campos.items()
        }
        registros.append(campos)

df = pd.DataFrame(registros)

print(f"Total de registros: {len(df):,}")
print(f"Colunas identificadas: {len(df.columns)}")

print("\n=== Tipos de log ===")
display(df["type"].value_counts(dropna=False).to_frame("quantidade"))

print("\n=== Subtipos ===")
display(df["subtype"].value_counts(dropna=False).head(15).to_frame("quantidade"))

print("\n=== Serviços mais frequentes ===")
display(df["service"].value_counts(dropna=False).head(15).to_frame("quantidade"))

print("\n=== Ações registradas ===")
display(df["action"].value_counts(dropna=False).head(15).to_frame("quantidade"))

### Etapa 4 — Validar o protocolo ICMP

In [ ]:
print("=== Protocolos mais frequentes ===")
display(
    df["proto"].value_counts(dropna=False).head(15).to_frame("quantidade")
)

print("\n=== Serviços por protocolo ===")
display(pd.crosstab(df["service"], df["proto"]).head(20))

print("\n=== Campos disponíveis para análise ===")
campos = [
    "date", "time", "srcip", "dstip", "proto", "service",
    "action", "sessionid", "duration", "sentbyte", "rcvdbyte",
    "sentpkt", "rcvdpkt", "logid"
]
print([campo for campo in campos if campo in df.columns])


### Etapa 5 — Criar a base de sessões ICMP

In [ ]:
# Filtrar tráfego ICMP aceito
icmp = df[
    (df["proto"].astype(str) == "1") &
    (df["action"] == "accept")
].copy()

# Converter campos numéricos
numericos = [
    "sessionid", "duration", "sentbyte",
    "rcvdbyte", "sentpkt", "rcvdpkt"
]

for coluna in numericos:
    if coluna in icmp.columns:
        icmp[coluna] = pd.to_numeric(
            icmp[coluna], errors="coerce"
        ).fillna(0)

# Ordenar e consolidar atualizações por sessão
chaves = ["srcip", "dstip", "sessionid"]

icmp = icmp.sort_values(
    ["eventtime"] if "eventtime" in icmp.columns else ["date", "time"]
    if "date" in icmp.columns and "time" in icmp.columns
    else ["sessionid"]
)

sessoes_icmp = (
    icmp.groupby(chaves, dropna=False, sort=False)
    .tail(1)
    .copy()
)

print("Registros ICMP aceitos:", len(icmp))
print("Sessões após consolidação:", len(sessoes_icmp))
print("Origens:", sessoes_icmp["srcip"].nunique())
print("Destinos:", sessoes_icmp["dstip"].nunique())

display(
    sessoes_icmp.sort_values("sentbyte", ascending=False)[
        ["srcip", "dstip", "sessionid", "duration",
         "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt"]
    ].head(10)
)


### Etapa 6 — Engenharia de atributos e Isolation Forest

In [ ]:
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler

# Criar atributos por sessão
features = sessoes_icmp[
    ["srcip", "dstip", "sessionid", "duration",
     "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt"]
].copy()

features["duration"] = features["duration"].clip(lower=1)

features["bytes_per_second"] = (
    features["sentbyte"] / features["duration"]
)

features["packets_per_second"] = (
    features["sentpkt"] / features["duration"]
)

features["bytes_per_packet"] = (
    features["sentbyte"] /
    features["sentpkt"].replace(0, np.nan)
).fillna(0)

features["byte_ratio"] = (
    features["sentbyte"] /
    features["rcvdbyte"].replace(0, np.nan)
).replace([np.inf, -np.inf], np.nan).fillna(0)

# Variáveis numéricas usadas pelo modelo
colunas_modelo = [
    "duration", "sentbyte", "rcvdbyte",
    "sentpkt", "rcvdpkt", "bytes_per_second",
    "packets_per_second", "bytes_per_packet", "byte_ratio"
]

X = features[colunas_modelo].replace(
    [np.inf, -np.inf], np.nan
).fillna(0)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

modelo = IsolationForest(
    n_estimators=200,
    contamination=0.01,
    random_state=42
)

features["anomaly_label"] = modelo.fit_predict(X_scaled)
features["anomaly_score"] = -modelo.decision_function(X_scaled)
features["is_anomaly"] = features["anomaly_label"] == -1

print("Sessões analisadas:", len(features))
print("Anomalias identificadas:", features["is_anomaly"].sum())

display(
    features.sort_values("anomaly_score", ascending=False)[
        ["srcip", "dstip", "sessionid", "sentbyte", "duration",
         "bytes_per_second", "packets_per_second",
         "anomaly_score", "is_anomaly"]
    ].head(20)
)

In [ ]:
# 1. Maiores sessões por volume enviado
print("MAIORES SESSÕES ICMP POR VOLUME ENVIADO")

display(
    features.sort_values("sentbyte", ascending=False)[
        [
            "srcip", "dstip", "sessionid",
            "sentbyte", "rcvdbyte", "duration",
            "bytes_per_second", "anomaly_score", "is_anomaly"
        ]
    ].head(15)
)

# 2. Anomalias por volume enviado
print("ANOMALIAS COM MAIOR VOLUME ENVIADO")

display(
    features[features["is_anomaly"]]
    .sort_values("sentbyte", ascending=False)[
        [
            "srcip", "dstip", "sessionid",
            "sentbyte", "rcvdbyte", "duration",
            "bytes_per_second", "anomaly_score"
        ]
    ].head(15)
)

### Etapa 7 - Validar as sessões suspeitas

In [ ]:
filtro = (
    (df["proto"].astype(str) == "1")
    & (df["srcip"] == "172.19.15.6")
    & (df["dstip"] == "192.168.15.8")
)

registros = df.loc[filtro].copy()

print("Registros encontrados:", len(registros))
print("Ações:", registros["action"].value_counts(dropna=False).to_dict())
print("Log IDs:", registros["logid"].value_counts(dropna=False).to_dict())

display(
    registros[
        [
            c for c in [
                "date", "time", "srcip", "dstip", "sessionid",
                "action", "logid", "duration", "sentbyte",
                "rcvdbyte", "sentpkt", "rcvdpkt", "identifier"
            ] if c in registros.columns
        ]
    ].head(20)
)

### Etapa 8 - Analisar todas as sessões desse par de IPs

In [ ]:
filtro = (
    (df["proto"].astype(str) == "1")
    & (df["srcip"] == "172.19.15.6")
    & (df["dstip"] == "192.168.15.8")
    & (df["action"] == "accept")
)

detalhes = df.loc[filtro].copy()

for coluna in ["duration", "sentbyte", "rcvdbyte", "sentpkt", "rcvdpkt", "identifier"]:
    detalhes[coluna] = pd.to_numeric(detalhes[coluna], errors="coerce")

# Considera apenas o registro final de cada sessão
finais = (
    detalhes.sort_values(["date", "time"])
    .groupby("sessionid", dropna=False)
    .tail(1)
)

display(
    finais[
        [
            "date", "time", "sessionid", "logid",
            "duration", "sentbyte", "rcvdbyte",
            "sentpkt", "rcvdpkt", "identifier"
        ]
    ].sort_values("sentbyte", ascending=False)
)